On va faire notre clustering sur les données issues de la PCA

In [1]:
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
import matplotlib.cm as cm
import geopandas as gpd

df_travail = pd.read_parquet('donnees_pca_pour_clustering.parquet')

df_numerique2 = (
    df_travail.select_dtypes(include=[np.number])
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

scaler2 = StandardScaler()
X2 = pd.DataFrame(
    scaler2.fit_transform(df_numerique2),
    columns=df_numerique2.columns,
    index=df_numerique2.index
)

gmm2 = GaussianMixture(n_components=2, covariance_type='full', random_state=42)
labels2 = gmm2.fit_predict(X2) + 1

df_gmm_map_2 = df_travail.reset_index()
df_gmm_map_2["codecommune"] = df_gmm_map_2["codecommune"].astype(str).str.zfill(5)
df_gmm_map_2["Nom_Cluster_GMM2"] = "Cluster " + labels2.astype(str)

if "france_communes" not in globals():
    france_communes = gpd.read_file(
        "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/communes.geojson"
    )

carte_gmm_2 = france_communes.merge(
    df_gmm_map_2,
    left_on="code",
    right_on="codecommune",
    how="inner",
)

categories2 = sorted(carte_gmm_2["Nom_Cluster_GMM2"].unique())
cmap2 = cm.get_cmap("plasma", len(categories2))

fig, ax = plt.subplots(1, 1, figsize=(15, 15), dpi=150)
carte_gmm_2.plot(
    column="Nom_Cluster_GMM2",
    ax=ax,
    categorical=True,
    categories=categories2,
    cmap=cmap2,
    legend=True,
    linewidth=0,
    edgecolor="none",
    legend_kwds={
        "title": "Clusters GMM",
        "loc": "upper left",
        "bbox_to_anchor": (1, 1),
        "frameon": False,
    },
)
ax.set_axis_off()
plt.title("Carte de France par GMM à 2 clusters", fontsize=18, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()


KeyError: 'codecommune'